Notebook 7 — Architecture Generalization: 1D vs 2D vs All-to-All

This is the corrected version. Do not use the earlier Ring-based Notebook 7.

The core scientific comparison is now:

1D Brick-Wall
2D rectangular nearest-neighbour Grid
All-to-All

The experiment deliberately keeps the entire methodology fixed except connectivity. It does not assume what the exponents should be.

Because this is a genuinely new architecture experiment, this notebook does perform fresh quantum simulations.


### Cell 1 — Setup


In [ ]:
# ============================================================
# NOTEBOOK 7 — ARCHITECTURE GENERALIZATION
# ============================================================
#
# Architectures:
#   1. 1D Brick-Wall
#   2. 2D Rectangular Nearest-Neighbour Grid
#   3. All-to-All
#
# Scientific question:
#
#   Does the empirical finite-depth tau_BP scaling persist
#   across qualitatively different circuit connectivities?
#
# IMPORTANT:
#   - Same observable
#   - Same initialization
#   - Same gradient estimator
#   - Same threshold
#   - Same n,k grids
#   - Same nominal layer definition
#   - Independent architecture-specific fits
#
# We DO NOT assume beforehand what c should be.
#
# ============================================================

%pip install -q pennylane "numpy<2"

import os
import time
import json
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pennylane as qml
import statsmodels.api as sm

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

# ------------------------------------------------------------
# Core methodology
# ------------------------------------------------------------

THRESHOLD = 1e-2
N_SAMPLES = 50
MAX_DEPTH = 30

# Primary fitting domain
TRAIN_N = [8, 10, 12]

# Architecture validation domain
HELDOUT_N = [14]

# Same k grid as the baseline where possible
K_BY_N = {
    8:  [2, 3, 4, 5, 6, 8],
    10: [2, 3, 4, 5, 6, 8, 10],
    12: [2, 3, 4, 5, 6, 8, 10, 12],
    14: [2, 3, 4, 5, 6, 8, 10, 12, 14],
}

ARCHITECTURES = [
    "brickwall_1d",
    "grid_2d",
    "all_to_all",
]

MASTER_SEED = 20260902

# ------------------------------------------------------------
# Output/checkpoint paths
# ------------------------------------------------------------

OUTPUT_DIR = "notebook7_architecture_generalization"

CHECKPOINT_DIR = os.path.join(
    OUTPUT_DIR,
    "checkpoints"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

CHECKPOINT_FILE = os.path.join(
    CHECKPOINT_DIR,
    "architecture_checkpoint.pkl"
)

RAW_OUTPUT = os.path.join(
    OUTPUT_DIR,
    "architecture_variance_curves.csv"
)

TAU_OUTPUT = os.path.join(
    OUTPUT_DIR,
    "architecture_tau.csv"
)

print("=" * 80)
print("NOTEBOOK 7 — ARCHITECTURE GENERALIZATION")
print("=" * 80)

print("PennyLane version:", qml.__version__)
print("Threshold:", THRESHOLD)
print("Samples:", N_SAMPLES)
print("Max depth:", MAX_DEPTH)
print("Architectures:", ARCHITECTURES)
print("Training n:", TRAIN_N)
print("Held-out n:", HELDOUT_N)
print("Master seed:", MASTER_SEED)
print("=" * 80)

### Cell 2 — Select simulator


In [ ]:
# ============================================================
# CELL 2 — SIMULATOR
# ============================================================

try:
    test_device = qml.device(
        "lightning.qubit",
        wires=2
    )

    SIMULATOR = "lightning.qubit"

except Exception:
    SIMULATOR = "default.qubit"

print(
    "Simulator selected:",
    SIMULATOR
)

### Cell 3 — Define deterministic 2D rectangular layouts


In [ ]:
# ============================================================
# CELL 3 — 2D RECTANGULAR GRID LAYOUTS
# ============================================================
#
# We use rectangular nearest-neighbour grids because the
# chosen n values are not all perfect squares.
#
# Layouts:
#
#   n=8  -> 2 x 4
#   n=10 -> 2 x 5
#   n=12 -> 3 x 4
#   n=14 -> 2 x 7
#
# Qubits are indexed row-major:
#
#   0  1  2  3 ...
#   ...
#
# The mapping is deterministic and reported explicitly.
# ============================================================

GRID_SHAPES = {
    8:  (2, 4),
    10: (2, 5),
    12: (3, 4),
    14: (2, 7),
}


def grid_edges(n_qubits):

    if n_qubits not in GRID_SHAPES:
        raise ValueError(
            f"No deterministic grid shape defined for n={n_qubits}."
        )

    rows, cols = GRID_SHAPES[n_qubits]

    if rows * cols != n_qubits:
        raise RuntimeError(
            "Grid shape does not match n."
        )

    edges = []

    # Horizontal nearest neighbours
    for r in range(rows):

        for c in range(cols - 1):

            q1 = r * cols + c
            q2 = r * cols + c + 1

            edges.append(
                (q1, q2)
            )

    # Vertical nearest neighbours
    for r in range(rows - 1):

        for c in range(cols):

            q1 = r * cols + c
            q2 = (r + 1) * cols + c

            edges.append(
                (q1, q2)
            )

    return edges


for n in sorted(GRID_SHAPES):

    edges = grid_edges(n)

    print(
        f"n={n}: shape={GRID_SHAPES[n]}, "
        f"nearest-neighbour edges={len(edges)}"
    )

### Cell 4 — Build matchings for 2D grid scheduling


In [ ]:
# ============================================================
# CELL 4 — GRID EDGE COLORING / MATCHING
# ============================================================
#
# A single circuit layer should not contain overlapping
# two-qubit gates if we interpret layer as a parallel
# entangling time slice.
#
# We therefore partition the grid edges into disjoint
# matchings.
#
# The matching decomposition is deterministic.
# ============================================================

def greedy_edge_matching(edges):

    remaining = list(edges)
    matchings = []

    while remaining:

        used_qubits = set()
        matching = []
        next_remaining = []

        for edge in remaining:

            q1, q2 = edge

            if (
                q1 not in used_qubits
                and q2 not in used_qubits
            ):

                matching.append(edge)
                used_qubits.add(q1)
                used_qubits.add(q2)

            else:

                next_remaining.append(edge)

        matchings.append(
            matching
        )

        remaining = next_remaining

    return matchings


GRID_MATCHINGS = {}

for n in GRID_SHAPES:

    edges = grid_edges(n)

    matchings = greedy_edge_matching(
        edges
    )

    GRID_MATCHINGS[n] = matchings

    print(
        f"\nn={n}: "
        f"{len(matchings)} grid entangling sublayers"
    )

    for idx, matching in enumerate(
        matchings
    ):

        print(
            f"  matching {idx}: "
            f"{matching}"
        )

### Cell 5 — Architecture entanglers


In [ ]:
# ============================================================
# CELL 5 — ARCHITECTURE-SPECIFIC ENTANGLERS
# ============================================================

def brickwall_entanglers(
    n_qubits,
    layer
):

    # Alternating 1D nearest-neighbour matching
    if layer % 2 == 0:

        pairs = [
            (q, q + 1)
            for q in range(
                0,
                n_qubits - 1,
                2
            )
        ]

    else:

        pairs = [
            (q, q + 1)
            for q in range(
                1,
                n_qubits - 1,
                2
            )
        ]

    for q1, q2 in pairs:

        qml.CNOT(
            wires=[q1, q2]
        )


def grid_2d_entanglers(
    n_qubits,
    layer
):

    matchings = GRID_MATCHINGS[
        n_qubits
    ]

    # Cycle deterministically through edge matchings.
    matching = matchings[
        layer % len(matchings)
    ]

    for q1, q2 in matching:

        qml.CNOT(
            wires=[q1, q2]
        )


def all_to_all_entanglers(
    n_qubits,
    layer
):
    """
    Complete-graph connectivity with a deterministic
    round-robin tournament schedule.

    Each round is a matching, so all-to-all connectivity
    is decomposed into parallel two-qubit time slices.

    This avoids placing overlapping CNOTs in the same
    nominal circuit layer.
    """

    # Circle-method round-robin schedule.
    nodes = list(range(n_qubits))

    # For odd n, add dummy node.
    dummy = None

    if len(nodes) % 2 == 1:

        nodes.append(None)
        dummy = None

    rounds = len(nodes) - 1

    # Fix one node and rotate the remainder.
    fixed = nodes[-1]
    rotating = nodes[:-1]

    round_matchings = []

    for r in range(rounds):

        left = [fixed] + rotating[:]
        pairs = []

        for i in range(
            len(left) // 2
        ):

            a = left[i]
            b = left[-1 - i]

            if (
                a is not None
                and b is not None
            ):
                pairs.append(
                    (a, b)
                )

        round_matchings.append(
            pairs
        )

        # Rotate while preserving fixed node.
        rotating = (
            [rotating[-1]]
            + rotating[:-1]
        )

    matching = round_matchings[
        layer % len(round_matchings)
    ]

    for q1, q2 in matching:

        qml.CNOT(
            wires=[q1, q2]
        )

### Cell 6 — Verify connectivity schedules


In [ ]:
# ============================================================
# CELL 6 — CONNECTIVITY SCHEDULE AUDIT
# ============================================================

def validate_matching(
    pairs
):

    used = set()

    for q1, q2 in pairs:

        if q1 == q2:
            return False

        if (
            q1 in used
            or q2 in used
        ):
            return False

        used.add(q1)
        used.add(q2)

    return True


print("=" * 80)
print("CONNECTIVITY SCHEDULE AUDIT")
print("=" * 80)

for n in sorted(
    K_BY_N.keys()
):

    # Brick-Wall
    for layer in range(2):

        if layer % 2 == 0:
            pairs = [
                (q, q + 1)
                for q in range(
                    0,
                    n - 1,
                    2
                )
            ]
        else:
            pairs = [
                (q, q + 1)
                for q in range(
                    1,
                    n - 1,
                    2
                )
            ]

        if not validate_matching(pairs):

            raise RuntimeError(
                f"Invalid Brick-Wall matching n={n}"
            )

    # Grid
    for matching in GRID_MATCHINGS[n]:

        if not validate_matching(
            matching
        ):

            raise RuntimeError(
                f"Invalid grid matching n={n}"
            )

print(
    "PASS: All local connectivity layers are valid matchings."
)

### Cell 7 — Common ansatz


In [ ]:
# ============================================================
# CELL 7 — COMMON HARDWARE-EFFICIENT ANSATZ
# ============================================================
#
# The ONLY architecture-dependent component is the
# entangling connectivity.
#
# Every layer has:
#   - n independent RY parameters
#   - one entangling matching
#
# Therefore:
#   n_params = n * depth
#
# ============================================================

def architecture_ansatz(
    architecture,
    n_qubits,
    params,
    depth
):

    expected_parameters = (
        n_qubits * depth
    )

    if len(params) != expected_parameters:

        raise ValueError(
            f"Expected {expected_parameters} parameters, "
            f"received {len(params)}."
        )

    index = 0

    for layer in range(depth):

        # -----------------------------------------------
        # Local rotations
        # -----------------------------------------------

        for q in range(n_qubits):

            qml.RY(
                params[index],
                wires=q
            )

            index += 1

        # -----------------------------------------------
        # Connectivity
        # -----------------------------------------------

        if architecture == "brickwall_1d":

            brickwall_entanglers(
                n_qubits,
                layer
            )

        elif architecture == "grid_2d":

            grid_2d_entanglers(
                n_qubits,
                layer
            )

        elif architecture == "all_to_all":

            all_to_all_entanglers(
                n_qubits,
                layer
            )

        else:

            raise ValueError(
                f"Unknown architecture: {architecture}"
            )

### Cell 8 — Observable


In [ ]:
# ============================================================
# CELL 8 — k-LOCAL OBSERVABLE
# ============================================================

def make_observable(
    k
):

    if k < 1:
        raise ValueError(
            "k must be >= 1."
        )

    observable = qml.PauliZ(
        0
    )

    for q in range(
        1,
        k
    ):

        observable = (
            observable
            @ qml.PauliZ(q)
        )

    return observable

### Cell 9 — QNode


In [ ]:
# ============================================================
# CELL 9 — QNODE FACTORY
# ============================================================

def make_qnode(
    architecture,
    n_qubits,
    k,
    depth
):

    device = qml.device(
        SIMULATOR,
        wires=n_qubits
    )

    observable = make_observable(
        k
    )

    diff_method = "adjoint" if "lightning" in SIMULATOR else "backprop"
    @qml.qnode(
        device,
        diff_method=diff_method
    )
    def cost_fn(
        params
    ):

        architecture_ansatz(
            architecture,
            n_qubits,
            params,
            depth
        )

        return qml.expval(
            observable
        )

    return cost_fn

### Cell 10 — Pooled variance estimator


In [ ]:
# ============================================================
# CELL 10 — POOLED GRADIENT VARIANCE
# ============================================================
#
# EXACT SAME ESTIMATOR FOR ALL ARCHITECTURES
#
# For each of S=50 independent parameter sets:
#   calculate complete gradient vector
#   append every gradient component
#
# Finally:
#
#   var_pooled = Var(all gradient components)
#
# ============================================================

def pooled_gradient_variance(
    architecture,
    n_qubits,
    k,
    depth,
    rng
):

    qnode = make_qnode(
        architecture,
        n_qubits,
        k,
        depth
    )

    all_gradients = []

    for sample in range(
        N_SAMPLES
    ):

        parameter_values = rng.uniform(
            0.0,
            2.0 * np.pi,
            size=n_qubits * depth
        )

        params = qml.numpy.array(
            parameter_values,
            requires_grad=True
        )

        gradient = qml.grad(
            qnode
        )(params)

        gradient = np.asarray(
            gradient,
            dtype=float
        ).reshape(-1)

        expected_gradient_length = (
            n_qubits * depth
        )

        if len(gradient) != expected_gradient_length:

            raise RuntimeError(
                "CRITICAL: Wrong gradient vector length."
            )

        all_gradients.extend(
            gradient.tolist()
        )

    all_gradients = np.asarray(
        all_gradients,
        dtype=float
    )

    expected_components = (
        N_SAMPLES
        * n_qubits
        * depth
    )

    if len(all_gradients) != expected_components:

        raise RuntimeError(
            "CRITICAL: Pooled component count mismatch."
        )

    variance = np.var(
        all_gradients
    )

    return (
        float(variance),
        int(len(all_gradients))
    )

### Cell 11 — Configuration list


In [ ]:
# ============================================================
# CELL 11 — CONFIGURATION GRID
# ============================================================

configurations = []

for architecture in ARCHITECTURES:

    for n in sorted(
        K_BY_N.keys()
    ):

        for k in K_BY_N[n]:

            configurations.append(
                (
                    architecture,
                    n,
                    k
                )
            )

print(
    f"Total configurations: {len(configurations)}"
)

for architecture in ARCHITECTURES:

    print(
        architecture,
        ":",
        sum(
            c[0] == architecture
            for c in configurations
        )
    )

print(
    "Total depth jobs:",
    len(configurations)
    * MAX_DEPTH
)

### Cell 12 — Seeds and checkpoint


In [ ]:
# ============================================================
# CELL 12 — INDEPENDENT CONFIGURATION SEEDS
# ============================================================

master_rng = np.random.default_rng(
    MASTER_SEED
)

config_seeds = {}

for configuration in configurations:

    config_seeds[
        configuration
    ] = int(
        master_rng.integers(
            0,
            2**32 - 1
        )
    )

print(
    "Independent seeds generated for all configurations."
)

### Cell 13 — Run fresh architecture simulations


In [ ]:
# ============================================================
# CELL 13 — MAIN FRESH SIMULATION
# ============================================================

print("=" * 80)
print("STARTING ARCHITECTURE SIMULATIONS")
print("=" * 80)

if os.path.exists(
    CHECKPOINT_FILE
):

    print(
        "Checkpoint found. Resuming."
    )

    with open(
        CHECKPOINT_FILE,
        "rb"
    ) as f:

        checkpoint = pickle.load(
            f
        )

    results = checkpoint["results"]

    completed = set(
        tuple(x)
        for x in checkpoint["completed"]
    )

    # Recover original seeds to ensure resume consistency.
    stored_seeds = checkpoint.get(
        "config_seeds",
        {}
    )

    for key, value in stored_seeds.items():

        config_seeds[
            tuple(key)
            if not isinstance(key, tuple)
            else key
        ] = value

else:

    results = []
    completed = set()

total_configs = len(
    configurations
)

for config_index, configuration in enumerate(
    configurations,
    start=1
):

    architecture, n, k = configuration

    if configuration in completed:

        print(
            f"[SKIP {config_index}/{total_configs}] "
            f"{architecture} | n={n} | k={k}"
        )

        continue

    print(
        "\n" + "=" * 75
    )

    print(
        f"CONFIGURATION "
        f"{config_index}/{total_configs}"
    )

    print(
        f"Architecture: {architecture}"
    )

    print(
        f"n={n}, k={k}"
    )

    print(
        f"Seed={config_seeds[configuration]}"
    )

    print(
        "=" * 75
    )

    rng = np.random.default_rng(
        config_seeds[configuration]
    )

    curve = []
    component_counts = []

    configuration_start = time.time()

    for depth in range(
        1,
        MAX_DEPTH + 1
    ):

        depth_start = time.time()

        variance, component_count = (
            pooled_gradient_variance(
                architecture,
                n,
                k,
                depth,
                rng
            )
        )

        curve.append(
            variance
        )

        component_counts.append(
            component_count
        )

        depth_elapsed = (
            time.time()
            - depth_start
        )

        if (
            depth == 1
            or depth % 5 == 0
            or depth == MAX_DEPTH
        ):

            print(
                f"depth {depth:2d}/{MAX_DEPTH}"
                f" | var={variance:.6e}"
                f" | components={component_count:,}"
                f" | {depth_elapsed:.2f}s"
            )

    crossings = [
        depth
        for depth, variance in enumerate(
            curve,
            start=1
        )
        if variance < THRESHOLD
    ]

    if crossings:

        tau = crossings[0]
        censored = False

    else:

        tau = MAX_DEPTH + 1
        censored = True

    elapsed = (
        time.time()
        - configuration_start
    )

    result = {
        "architecture":
            architecture,

        "n":
            n,

        "k":
            k,

        "threshold":
            THRESHOLD,

        "n_samples":
            N_SAMPLES,

        "max_depth":
            MAX_DEPTH,

        "configuration_seed":
            config_seeds[configuration],

        "tau_BP":
            tau,

        "censored":
            censored,

        "runtime_s":
            elapsed,

        "variance_curve":
            curve,

        "n_grad_components":
            component_counts,
    }

    results.append(
        result
    )

    completed.add(
        configuration
    )

    checkpoint_payload = {
        "results":
            results,

        "completed":
            list(completed),

        "config_seeds":
            config_seeds,

        "protocol": {
            "threshold":
                THRESHOLD,

            "n_samples":
                N_SAMPLES,

            "max_depth":
                MAX_DEPTH,

            "architectures":
                ARCHITECTURES,

            "K_BY_N":
                K_BY_N,

            "training_n":
                TRAIN_N,

            "heldout_n":
                HELDOUT_N,

            "simulator":
                SIMULATOR,

            "master_seed":
                MASTER_SEED,
        }
    }

    with open(
        CHECKPOINT_FILE,
        "wb"
    ) as f:

        pickle.dump(
            checkpoint_payload,
            f
        )

    print(
        f"\nFINISHED "
        f"{architecture}, n={n}, k={k}"
    )

    print(
        f"tau_BP = {tau}"
    )

    print(
        f"Censored = {censored}"
    )

    print(
        f"Runtime = {elapsed:.1f}s"
    )

    print(
        "[CHECKPOINT SAVED]"
    )

### Cell 14 — Build raw architecture dataset


In [ ]:
# ============================================================
# CELL 14 — RAW CURVES
# ============================================================

raw_records = []

for result in results:

    for depth, (
        variance,
        component_count
    ) in enumerate(
        zip(
            result["variance_curve"],
            result["n_grad_components"]
        ),
        start=1
    ):

        raw_records.append({
            "framework":
                "PennyLane",

            "framework_version":
                qml.__version__,

            "simulator":
                SIMULATOR,

            "architecture":
                result["architecture"],

            "n":
                result["n"],

            "k":
                result["k"],

            "depth":
                depth,

            "n_params":
                result["n"] * depth,

            "n_param_sets":
                N_SAMPLES,

            "n_grad_components":
                component_count,

            "var_pooled":
                variance,

            "threshold":
                THRESHOLD,

            "configuration_seed":
                result["configuration_seed"],
        })

architecture_raw_df = pd.DataFrame(
    raw_records
)

architecture_raw_df = (
    architecture_raw_df
    .sort_values(
        [
            "architecture",
            "n",
            "k",
            "depth",
        ]
    )
    .reset_index(drop=True)
)

architecture_raw_df.to_csv(
    RAW_OUTPUT,
    index=False
)

print(
    f"Saved {len(architecture_raw_df):,} raw observations."
)

print(
    RAW_OUTPUT
)

### Cell 15 — Reconstruct \(\tau_{BP}\)


In [ ]:
# ============================================================
# CELL 15 — TAU_BP RECONSTRUCTION
# ============================================================

tau_records = []

for (
    architecture,
    n,
    k
), group in architecture_raw_df.groupby(
    [
        "architecture",
        "n",
        "k",
    ]
):

    group = group.sort_values(
        "depth"
    )

    crossed = group[
        group["var_pooled"] < THRESHOLD
    ]

    if len(crossed) > 0:

        tau = int(
            crossed.iloc[0]["depth"]
        )

        censored = False

    else:

        tau = MAX_DEPTH + 1
        censored = True

    tau_records.append({
        "architecture":
            architecture,

        "n":
            int(n),

        "k":
            int(k),

        "nk":
            int(n * k),

        "tau_BP":
            tau,

        "censored":
            censored,
    })

architecture_tau_df = pd.DataFrame(
    tau_records
)

architecture_tau_df = (
    architecture_tau_df
    .sort_values(
        [
            "architecture",
            "n",
            "k",
        ]
    )
    .reset_index(drop=True)
)

architecture_tau_df.to_csv(
    TAU_OUTPUT,
    index=False
)

display(
    architecture_tau_df
)

### Cell 16 — Coverage and methodology audit


In [ ]:
# ============================================================
# CELL 16 — COVERAGE / ESTIMATOR AUDIT
# ============================================================

print("=" * 80)
print("ARCHITECTURE COVERAGE AUDIT")
print("=" * 80)

for architecture in ARCHITECTURES:

    sub = architecture_tau_df[
        architecture_tau_df[
            "architecture"
        ] == architecture
    ]

    print(
        f"{architecture}: "
        f"{len(sub)} configurations, "
        f"{sub['censored'].sum()} censored"
    )

# ------------------------------------------------------------
# Configuration coverage
# ------------------------------------------------------------

reference_set = None

for architecture in ARCHITECTURES:

    sub = architecture_tau_df[
        architecture_tau_df[
            "architecture"
        ] == architecture
    ]

    current_set = set(
        zip(
            sub["n"],
            sub["k"]
        )
    )

    if reference_set is None:

        reference_set = current_set

    elif current_set != reference_set:

        raise RuntimeError(
            "CRITICAL: Architecture coverage differs."
        )

print(
    "\nPASS: All architectures use identical (n,k) coverage."
)

# ------------------------------------------------------------
# Pooled component counts
# ------------------------------------------------------------

expected_components = (
    N_SAMPLES
    * architecture_raw_df["n"]
    * architecture_raw_df["depth"]
)

bad_counts = (
    architecture_raw_df[
        "n_grad_components"
    ].to_numpy()
    != expected_components.to_numpy()
)

if bad_counts.any():

    raise RuntimeError(
        "CRITICAL: Pooled gradient component counts failed."
    )

print(
    "PASS: Pooled gradient estimator count is consistent."
)

### Cell 17 — Fit the same law separately


In [ ]:
# ============================================================
# CELL 17 — ARCHITECTURE-SPECIFIC FITS
# ============================================================
#
# SAME empirical model:
#
#     tau_BP = A * (n*k)^(-c)
#
# Each architecture receives its own fit.
#
# ONLY n=8,10,12 are used for these fits.
# n=14 is reserved for architecture-specific validation.
# ============================================================

def fit_product_model(
    data
):

    usable = data[
        (~data["censored"])
        &
        (data["n"].isin(TRAIN_N))
    ].copy()

    if len(usable) < 3:

        return None

    usable["log_tau"] = np.log(
        usable["tau_BP"].astype(float)
    )

    usable["log_nk"] = np.log(
        usable["nk"].astype(float)
    )

    X = sm.add_constant(
        usable["log_nk"]
    )

    model = sm.OLS(
        usable["log_tau"],
        X
    ).fit()

    log_A = float(
        model.params["const"]
    )

    slope = float(
        model.params["log_nk"]
    )

    A = float(
        np.exp(log_A)
    )

    c = float(
        -slope
    )

    prediction = (
        A
        * usable["nk"].astype(float)
        .pow(-c)
    )

    residuals = (
        usable["tau_BP"].astype(float)
        - prediction
    )

    mae = mean_absolute_error(
        usable["tau_BP"],
        prediction
    )

    rmse = np.sqrt(
        mean_squared_error(
            usable["tau_BP"],
            prediction
        )
    )

    r2_tau = r2_score(
        usable["tau_BP"],
        prediction
    )

    # Correct finite-sample AICc
    n_obs = len(usable)
    p = int(model.df_model + 1)

    if n_obs > p + 1:

        aicc = (
            model.aic
            + (
                2 * p * (p + 1)
                /
                (n_obs - p - 1)
            )
        )

    else:

        aicc = np.nan

    return {
        "model":
            model,

        "training_data":
            usable,

        "A":
            A,

        "c":
            c,

        "A_SE":
            float(
                np.exp(
                    log_A
                )
                * model.bse["const"]
            ),

        "c_SE":
            float(
                model.bse["log_nk"]
            ),

        "c_CI_low":
            float(
                -model.conf_int()
                .loc["log_nk", 1]
            ),

        "c_CI_high":
            float(
                -model.conf_int()
                .loc["log_nk", 0]
            ),

        "MAE":
            float(mae),

        "RMSE":
            float(rmse),

        "R2_tau":
            float(r2_tau),

        "R2_log":
            float(model.rsquared),

        "AIC":
            float(model.aic),

        "AICc":
            float(aicc),

        "BIC":
            float(model.bic),
    }


fit_objects = {}
fit_rows = []

for architecture in ARCHITECTURES:

    sub = architecture_tau_df[
        architecture_tau_df[
            "architecture"
        ] == architecture
    ].copy()

    result = fit_product_model(
        sub
    )

    if result is None:

        continue

    fit_objects[
        architecture
    ] = result

    fit_rows.append({
        "architecture":
            architecture,

        "A":
            result["A"],

        "c":
            result["c"],

        "c_SE":
            result["c_SE"],

        "c_CI_low":
            result["c_CI_low"],

        "c_CI_high":
            result["c_CI_high"],

        "R2_log":
            result["R2_log"],

        "R2_tau":
            result["R2_tau"],

        "MAE_train":
            result["MAE"],

        "RMSE_train":
            result["RMSE"],

        "AIC":
            result["AIC"],

        "AICc":
            result["AICc"],

        "BIC":
            result["BIC"],

        "n_training":
            len(
                result[
                    "training_data"
                ]
            ),
    })

architecture_fit_df = pd.DataFrame(
    fit_rows
)

print("=" * 80)
print("ARCHITECTURE-SPECIFIC FITS")
print("=" * 80)

display(
    architecture_fit_df
)

### Cell 18 — Frozen \(n=14\) validation for every architecture


In [ ]:
# ============================================================
# CELL 18 — HELD-OUT ARCHITECTURE VALIDATION
# ============================================================
#
# Each architecture's A and c are frozen after its n=8,10,12 fit.
#
# n=14 is then predicted without refitting.
# ============================================================

validation_rows = []

for architecture in ARCHITECTURES:

    result = fit_objects[
        architecture
    ]

    A = result["A"]
    c = result["c"]

    heldout = architecture_tau_df[
        (
            architecture_tau_df[
                "architecture"
            ] == architecture
        )
        &
        (
            architecture_tau_df[
                "n"
            ].isin(
                HELDOUT_N
            )
        )
        &
        (
            ~architecture_tau_df[
                "censored"
            ]
        )
    ].copy()

    heldout[
        "tau_pred"
    ] = (
        A
        * heldout["nk"].astype(float)
        .pow(-c)
    )

    heldout[
        "residual"
    ] = (
        heldout["tau_BP"]
        - heldout["tau_pred"]
    )

    heldout[
        "absolute_error"
    ] = (
        heldout["residual"]
        .abs()
    )

    y_true = (
        heldout["tau_BP"]
        .astype(float)
        .to_numpy()
    )

    y_pred = (
        heldout["tau_pred"]
        .astype(float)
        .to_numpy()
    )

    residual = (
        y_true - y_pred
    )

    validation_rows.append({
        "architecture":
            architecture,

        "N_heldout":
            len(heldout),

        "MAE":
            mean_absolute_error(
                y_true,
                y_pred
            ),

        "RMSE":
            np.sqrt(
                mean_squared_error(
                    y_true,
                    y_pred
                )
            ),

        "MAPE_percent":
            np.mean(
                np.abs(
                    residual / y_true
                )
            ) * 100,

        "R2":
            r2_score(
                y_true,
                y_pred
            ),

        "mean_signed_error":
            np.mean(
                residual
            ),

        "within_1_layer_percent":
            np.mean(
                np.abs(residual) <= 1
            ) * 100,

        "within_2_layers_percent":
            np.mean(
                np.abs(residual) <= 2
            ) * 100,
    })

    heldout.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"heldout_{architecture}.csv"
        ),
        index=False
    )

architecture_validation_df = pd.DataFrame(
    validation_rows
)

print("=" * 80)
print("ARCHITECTURE-SPECIFIC HELD-OUT n=14 VALIDATION")
print("=" * 80)

display(
    architecture_validation_df
)

### Cell 19 — Compare exponents statistically


In [ ]:
# ============================================================
# CELL 19 — EXPONENT COMPARISON
# ============================================================

print("=" * 80)
print("PAIRWISE EXPONENT COMPARISON")
print("=" * 80)

pairwise_rows = []

for i in range(
    len(architecture_fit_df)
):

    for j in range(
        i + 1,
        len(architecture_fit_df)
    ):

        a = architecture_fit_df.iloc[i]
        b = architecture_fit_df.iloc[j]

        delta_c = (
            a["c"]
            - b["c"]
        )

        pooled_se = np.sqrt(
            a["c_SE"] ** 2
            +
            b["c_SE"] ** 2
        )

        if pooled_se > 0:

            z = (
                delta_c
                /
                pooled_se
            )

            # Two-sided normal approximation.
            from scipy.stats import norm

            p_value = (
                2
                * norm.sf(
                    abs(z)
                )
            )

        else:

            z = np.nan
            p_value = np.nan

        pairwise_rows.append({
            "architecture_1":
                a["architecture"],

            "architecture_2":
                b["architecture"],

            "c_1":
                a["c"],

            "c_2":
                b["c"],

            "delta_c":
                delta_c,

            "approx_SE_delta_c":
                pooled_se,

            "z":
                z,

            "p_value":
                p_value,
        })

pairwise_c_df = pd.DataFrame(
    pairwise_rows
)

display(
    pairwise_c_df
)

### Cell 20 — Scaling curves by architecture


In [ ]:
# ============================================================
# CELL 20 — SCALING CURVES
# ============================================================

plt.figure(
    figsize=(10, 7)
)

for architecture in ARCHITECTURES:

    sub = architecture_tau_df[
        (
            architecture_tau_df[
                "architecture"
            ] == architecture
        )
        &
        (
            architecture_tau_df[
                "n"
            ].isin(
                TRAIN_N
            )
        )
        &
        (
            ~architecture_tau_df[
                "censored"
            ]
        )
    ]

    plt.scatter(
        sub["nk"],
        sub["tau_BP"],
        s=55,
        label=architecture
    )

    fit = fit_objects[
        architecture
    ]

    nk_grid = np.geomspace(
        sub["nk"].min(),
        sub["nk"].max(),
        200
    )

    tau_grid = (
        fit["A"]
        * nk_grid
        ** (-fit["c"])
    )

    plt.plot(
        nk_grid,
        tau_grid,
        linewidth=2
    )

plt.xscale("log")
plt.yscale("log")

plt.xlabel(
    "n × k"
)

plt.ylabel(
    r"$\tau_{BP}$"
)

plt.title(
    "Architecture Dependence of Finite-Depth BP Onset"
)

plt.grid(
    True,
    alpha=0.3
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "architecture_scaling.png"
    ),
    dpi=250,
    bbox_inches="tight"
)

plt.show()

### Cell 21 — Exponent comparison


In [ ]:
# ============================================================
# CELL 21 — EXPONENT PLOT
# ============================================================

plt.figure(
    figsize=(9, 6)
)

plt.bar(
    architecture_fit_df[
        "architecture"
    ],
    architecture_fit_df[
        "c"
    ]
)

plt.ylabel(
    "Fitted exponent c"
)

plt.xlabel(
    "Architecture"
)

plt.title(
    "Empirical Scaling Exponent by Connectivity"
)

plt.grid(
    True,
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "architecture_exponents.png"
    ),
    dpi=250,
    bbox_inches="tight"
)

plt.show()

### Cell 22 — Held-out prediction comparison


In [ ]:
# ============================================================
# CELL 22 — HELD-OUT PREDICTION COMPARISON
# ============================================================

plt.figure(
    figsize=(10, 7)
)

for architecture in ARCHITECTURES:

    path = os.path.join(
        OUTPUT_DIR,
        f"heldout_{architecture}.csv"
    )

    if not os.path.exists(path):
        continue

    sub = pd.read_csv(
        path
    )

    plt.scatter(
        sub["tau_BP"],
        sub["tau_pred"],
        s=65,
        label=architecture
    )

all_values = []

for architecture in ARCHITECTURES:

    path = os.path.join(
        OUTPUT_DIR,
        f"heldout_{architecture}.csv"
    )

    if not os.path.exists(path):
        continue

    sub = pd.read_csv(
        path
    )

    all_values.extend(
        sub["tau_BP"].tolist()
    )

    all_values.extend(
        sub["tau_pred"].tolist()
    )

if all_values:

    lo = np.floor(
        min(all_values)
    )

    hi = np.ceil(
        max(all_values)
    )

    plt.plot(
        [lo, hi],
        [lo, hi],
        linestyle="--",
        linewidth=2,
        label="Perfect prediction"
    )

plt.xlabel(
    "Observed tau_BP"
)

plt.ylabel(
    "Frozen predicted tau_BP"
)

plt.title(
    "Architecture-Specific Frozen Prediction at n=14"
)

plt.grid(
    True,
    alpha=0.3
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "architecture_heldout_predictions.png"
    ),
    dpi=250,
    bbox_inches="tight"
)

plt.show()

### Cell 23 — Save final publication package


In [ ]:
# ============================================================
# CELL 23 — SAVE COMPLETE RESULTS
# ============================================================

architecture_fit_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "architecture_fit_summary.csv"
    ),
    index=False
)

architecture_validation_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "architecture_validation_summary.csv"
    ),
    index=False
)

pairwise_c_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "architecture_pairwise_c_comparison.csv"
    ),
    index=False
)

architecture_tau_df.to_csv(
    TAU_OUTPUT,
    index=False
)

architecture_raw_df.to_csv(
    RAW_OUTPUT,
    index=False
)

protocol = {
    "checkpoint": 7,

    "experiment":
        "architecture_generalization",

    "architectures":
        ARCHITECTURES,

    "threshold":
        THRESHOLD,

    "n_samples":
        N_SAMPLES,

    "max_depth":
        MAX_DEPTH,

    "training_n":
        TRAIN_N,

    "heldout_n":
        HELDOUT_N,

    "K_BY_N":
        K_BY_N,

    "simulator":
        SIMULATOR,

    "pennylane_version":
        qml.__version__,

    "parameter_distribution":
        "Uniform(0, 2*pi)",

    "observable":
        "Z_0 tensor ... tensor Z_(k-1)",

    "gradient_estimator":
        "Pool all complete gradient components "
        "from all parameter sets and calculate one variance",

    "tau_definition":
        "First depth where var_pooled < 1e-2",

    "grid_shapes":
        GRID_SHAPES,

    "master_seed":
        MASTER_SEED,
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "architecture_protocol.json"
    ),
    "w"
) as f:

    json.dump(
        protocol,
        f,
        indent=2
    )

print("=" * 80)
print("CHECKPOINT 7 PACKAGE SAVED")
print("=" * 80)

for fname in sorted(
    os.listdir(OUTPUT_DIR)
):

    print(
        os.path.join(
            OUTPUT_DIR,
            fname
        )
    )

print(
    "\nSTATUS: COMPLETE"
)

One critical methodological point

This notebook intentionally does not normalize depth by topology yet. First we need the raw experimental evidence. We want to know whether the different architectures produce different \(\tau_{BP}\) behavior under the same nominal circuit-layer definition.

Only after seeing the actual results should we investigate whether differences can be explained by causal-cone expansion or a topology-specific effective depth. That keeps the architecture experiment from becoming circular.

Also, the All-to-All implementation uses parallel round-robin matchings, rather than putting overlapping CNOTs into one circuit layer. This makes “depth” a meaningful parallel-circuit time-slice quantity instead of silently making one all-to-all layer contain \(O(n^2)\) sequential operations.

Run it until completion. The key outputs to send me are Cell 17, Cell 18, and Cell 19.
